In [1]:
# module to indicesise data (Jan 2020 as base 100)
import pandas as pd
import numpy as np


In [ ]:

def indexise(data, yoycolnum = 1, momcolnum = 2, base = "2020-01-01", percentage = True):
    data['Date'] = pd.to_datetime(data['Date'])
    data = data.sort_values('Date')
    
    data_yoy = data.iloc[:, [0, yoycolnum]].dropna()
    data_mom = data.iloc[:, [0, momcolnum]].dropna()
    date_end = data['Date'].max()
    date_yoy_start = data_yoy['Date'].min() - pd.DateOffset(months=12)
    date_mom_start = data_mom['Date'].min() - pd.DateOffset(months=1)

    # indice dataframe initialisation
    indice = pd.DataFrame(columns=['Date', 'Index Y', 'Index M', 'Index'])
    col_rec = len(indice.columns) - 2
    indice['Date'] = pd.date_range(start=min(date_yoy_start, date_mom_start), end=date_end, freq='MS')
    indice['Date'] = pd.to_datetime(indice['Date'])
    indice = indice.merge(data, on='Date', how='left')
    # after merging make sure yoycolnum and momcolnum still work
    indice = indice.set_index('Date')
    indice.loc[base, ["Index Y", "Index M", "Index"]] = 100

    # base year computation
    base_index = indice.index.get_loc(pd.Timestamp(base))
    base_next_index = base_index + 12
    for i in range(base_index + 1, base_next_index):
        # index = previous index cum mom
        indice.iloc[i, 2] = indice.iloc[i - 1, 2] * (1 + indice.iloc[i, momcolnum+col_rec] / 100)

    # computation of first data following base year
    # indice.iloc[base_next_index, 0] = indice.iloc[base_index, 2] * (1 + indice.iloc[base_next_index, yoycolnum+col_rec] / 100) # yoy
    # indice.iloc[base_next_index, 1] = indice.iloc[base_next_index - 1, 2] * (1 + indice.iloc[base_next_index, momcolnum+col_rec] / 100) # mom
    # indice.iloc[base_next_index, 2] = indice.iloc[base_next_index, [0, 1]].mean()

    # computation of following years
    for i in range(base_next_index, len(indice)):
        # index_y = previous year, same month index cum yoy
        if pd.isna(indice.iloc[i, yoycolnum+col_rec]):
            indice.iloc[i, 0] = pd.NA
        else:
            indice.iloc[i, 0] = indice.iloc[i - 12, 2] * (1 + indice.iloc[i, yoycolnum+col_rec] / 100)

        # index_m = previous month index cum mom
        if pd.isna(indice.iloc[i, momcolnum+col_rec]):
            indice.iloc[i, 1] = pd.NA
        else:
            indice.iloc[i, 1] = indice.iloc[i - 1, 2] * (1 + indice.iloc[i, momcolnum+col_rec] / 100)

        # index = average of index_y and index_m
        indice.iloc[i, 2] = indice.iloc[i, [0, 1]].mean()

    # computation of preceding years
    for i in range(base_index - 1, -1,-1):
        # index_y = following year, same month index subtract yoy
        if pd.isna(indice.iloc[i + 12, yoycolnum+col_rec]):
            indice.iloc[i, 0] = pd.NA
        else:
            indice.iloc[i, 0] = indice.iloc[i + 12, 2] / (1 + indice.iloc[i + 12, yoycolnum+col_rec] / 100)

        # index_m = following month index subtract mom
        if pd.isna(indice.iloc[i + 1, momcolnum+col_rec]):
            indice.iloc[i, 1] = pd.NA
        else:
            indice.iloc[i, 1] = indice.iloc[i + 1, 2] / (1 + indice.iloc[i + 1, momcolnum+col_rec] / 100)

        # index = average of index_y and index_m
        indice.iloc[i, 2] = indice.iloc[i, [0, 1]].mean()

    return indice['Index']


In [3]:
data = pd.read_csv('CN CPI (%).csv', date_format='%Y-%m-%d')
data = data[['Date', 'CPI(YOY)', 'CPI(MOM)']]
data['Date'].min()

'1987-01-01'

In [8]:
res = indexise(data)
res[res.index.year<=1987].head(24)

Date
1986-01-01    20.210615
1986-02-01    19.870813
1986-03-01    20.218825
1986-04-01    20.269257
1986-05-01     20.12027
1986-06-01    20.006051
1986-07-01    19.643556
1986-08-01    19.476821
1986-09-01     20.07939
1986-10-01    20.411107
1986-11-01    20.535691
1986-12-01    20.326944
1987-01-01    21.241357
1987-02-01    20.943837
1987-03-01    21.391517
1987-04-01    21.627297
1987-05-01    21.649411
1987-06-01    21.566523
1987-07-01    21.175753
1987-08-01     21.07392
1987-09-01    21.625503
1987-10-01     21.94194
1987-11-01    22.240153
1987-12-01    22.136042
Name: Index, dtype: object

In [9]:
res.tail(24)

Date
2024-05-01    102.042696
2024-06-01    101.746579
2024-07-01    102.205366
2024-08-01    102.602938
2024-09-01    102.624197
2024-10-01    102.383543
2024-11-01    101.798177
2024-12-01    101.823989
2025-01-01    102.535613
2025-02-01    102.322139
2025-03-01    101.944642
2025-04-01    102.069761
2025-05-01    101.903137
2025-06-01     101.82478
2025-07-01    102.218722
2025-08-01    102.205624
2025-09-01    102.312077
2025-10-01    102.552506
2025-11-01    102.480359
2025-12-01     102.66195
2026-01-01    102.803979
2026-02-01    103.742173
2026-03-01    102.990033
2026-04-01    103.296801
Name: Index, dtype: object

In [10]:
res.to_csv('res.csv')